In [1]:
import pandas as pd
import json
import datetime

In [32]:
cities = pd.read_csv('../dataset/cities.csv', sep=',')
airlines = pd.read_csv('../dataset/airlines.csv', sep=',')
flights = pd.read_csv('../dataset/flights.csv', sep=',')
logs = pd.read_csv('../dataset/logs.csv', sep=',')
airports = pd.read_csv('../dataset/airports.csv', sep=',')

In [17]:
print("cities shape:", cities.shape)
print("airlines shape:", airlines.shape)
print("flights shape:", flights.shape)
print("logs shape:", logs.shape)
print("airports shape:", airports.shape)

cities shape: (316, 9)
airlines shape: (19, 2)
flights shape: (5262836, 20)
logs shape: (5765, 10)
airports shape: (342, 7)


In [18]:
print(cities.columns)
print(airlines.columns)
print(flights.columns)
print(logs.columns)
print(airports.columns)

Index(['city', 'city_ascii', 'state_id', 'state_name', 'lat', 'lng',
       'population', 'density', 'timezone'],
      dtype='str')
Index(['IATA_CODE', 'AIRLINE'], dtype='str')
Index(['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER',
       'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE',
       'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'SCHEDULED_TIME', 'ELAPSED_TIME',
       'AIR_TIME', 'DISTANCE', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME',
       'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED'],
      dtype='str')
Index(['flight', 'lat', 'lon', 'alt', 'gspeed', 'vspeed', 'timestamp',
       'orig_iata', 'dest_iata', 'eta'],
      dtype='str')
Index(['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY', 'LATITUDE',
       'LONGITUDE'],
      dtype='str')


In [20]:
print(airports['LATITUDE'].dtype)
print(airports['LONGITUDE'].dtype)

float64
float64


In [ ]:
airports.query("IATA_CODE == 'AKL'")

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
13,AKL,Auckland Airport,Auckland,NaN,New Zealand,-3.70082,17.47916


In [17]:
airports.isna().sum()

IATA_CODE    0
AIRPORT      0
CITY         0
STATE        1
COUNTRY      0
LATITUDE     3
LONGITUDE    3
dtype: int64

In [26]:
flights.isna().sum()

YEAR                         0
MONTH                        0
DAY                          0
DAY_OF_WEEK                  0
AIRLINE                      0
FLIGHT_NUMBER                0
ORIGIN_AIRPORT               0
DESTINATION_AIRPORT          0
SCHEDULED_DEPARTURE          0
DEPARTURE_TIME         1597629
DEPARTURE_DELAY        1597629
SCHEDULED_TIME               2
ELAPSED_TIME             38999
AIR_TIME                 38999
DISTANCE                     0
SCHEDULED_ARRIVAL            0
ARRIVAL_TIME           1597791
ARRIVAL_DELAY          1606085
DIVERTED                     0
CANCELLED                    0
dtype: int64

In [27]:
flights.shape

(5262836, 20)

# File JSON airports

In [37]:
import pandas as pd
import json
import datetime

FILE_AIRLINES = '../dataset/airlines.csv'
FILE_AIRPORTS = '../dataset/airports.csv'
FILE_FLIGHTS = '../dataset/flights.csv'

OUTPUT_AIRPORTS = '../dataset/json/airports.json'
OUTPUT_FLIGHTS = '../dataset/json/flights.json'

print("1. Caricamento Lookup Tables (Airlines & Airports)...")

# 1. Carichiamo AIRLINES in un dizionario
airlines_df = pd.read_csv(FILE_AIRLINES, dtype=str)
airlines_map = dict(zip(airlines_df['IATA_CODE'], airlines_df['AIRLINE']))

# 2. Carichiamo AIRPORTS
airports_df = pd.read_csv(FILE_AIRPORTS, dtype=str)
airports_map = {}
airports_collection_data = []

counter = 0

for _, row in airports_df.iterrows():
    # if counter == 5:
    #     break
    try:
        lat = float(row['LATITUDE'])
        lon = float(row['LONGITUDE'])
        
        # Oggetto GeoJSON standard
        geo_location = {
            "type": "Point",
            "coordinates": [lon, lat] if pd.notnull(lat) and pd.notnull(lon) else None
        }
        
        airport_obj = {
            "_id": row['IATA_CODE'],
            "name": row['AIRPORT'],
            "city": row['CITY'],
            "state": row['STATE'] if pd.notnull(row['STATE']) else row['COUNTRY'],
            "country": row['COUNTRY'],
            "location": geo_location
        }
        
        airports_map[row['IATA_CODE']] = airport_obj
        airports_collection_data.append(airport_obj)
        
    except (ValueError, TypeError):
        print(f"Warning: Coordinate non valida per aeroporto {row['IATA_CODE']} - Skipping")
        continue

    counter += 1

# --- MODIFICA QUI ---
# Scriviamo il file JSON come un unico array valido
# with open(OUTPUT_AIRPORTS, 'w', encoding='utf-8') as f:
#     json.dump(airports_collection_data, f, ensure_ascii=False)

print(f"   -> Creato {OUTPUT_AIRPORTS} con {len(airports_collection_data)} aeroporti.")

1. Caricamento Lookup Tables (Airlines & Airports)...
   -> Creato ../dataset/json/airports.json con 342 aeroporti.


# Flights JSON

In [38]:
print("2. Elaborazione Voli e Denormalizzazione (Questo richiederà tempo)...")

# 3. Processiamo i VOLI in streaming (chunksize) per non intasare la RAM
count = 0
is_first_record = True # Flag per gestire la virgola

with open(OUTPUT_FLIGHTS, 'w') as f_out:
    f_out.write('[') # 1. Apriamo l'array JSON all'inizio del file
    
    # Leggiamo il CSV a blocchi
    for chunk in pd.read_csv(FILE_FLIGHTS, dtype=str, chunksize=100000):
        for _, row in chunk.iterrows():
            
            # Recuperiamo i dati dai lookup
            airline_code = row['AIRLINE']
            origin_code = row['ORIGIN_AIRPORT']
            dest_code = row['DESTINATION_AIRPORT']

            # Se l'aeroporto non esiste nel lookup (es. codici vecchi), saltiamo o gestiamo
            origin_data = airports_map.get(origin_code)
            dest_data = airports_map.get(dest_code)
            
            if not origin_data or not dest_data:
                # print(f"   Attenzione: Aeroporto non trovato per volo {row['FLIGHT_NUMBER']} - ORIGIN: {origin_code}, DEST: {dest_code}. Skipping...")
                continue

            # Costruzione della Data
            try:
                flight_date = datetime.datetime(
                    int(row['YEAR']), int(row['MONTH']), int(row['DAY'])
                ).isoformat()
            except:
                flight_date = None

            # --- COSTRUZIONE DEL DOCUMENTO DENORMALIZZATO ---
            doc = {
                # Dati Volo
                "flight_info": {
                    "airline_code": airline_code,
                    "airline_name": airlines_map.get(airline_code, "Unknown"),
                    "flight_number": row['FLIGHT_NUMBER'],
                    "date": flight_date,
                    "scheduled_departure": row['SCHEDULED_DEPARTURE'],
                    "scheduled_arrival": row['SCHEDULED_ARRIVAL'],
                    "scheduled_flight_duration": float(row['SCHEDULED_TIME']) if pd.notnull(row['SCHEDULED_TIME']) else None
                },
                
                # Dati Rotta (EMBEDDED AIRPORTS)
                "route": {
                    "origin": {
                        "iata": origin_code,
                        "airport_name": origin_data['name'],
                        "city": origin_data['city'],
                        "state": origin_data['state'],
                        "location": origin_data['location']
                    },
                    "destination": {
                        "iata": dest_code,
                        "airport_name": dest_data['name'],
                        "city": dest_data['city'],
                        "state": dest_data['state'],
                        "location": dest_data['location']
                    },
                    "distance": float(row['DISTANCE']) if pd.notnull(row['DISTANCE']) else None
                },
                
                # Dati Performance
                "stats": {
                    "tot_delay": float(row['DEPARTURE_DELAY']) + float(row['ARRIVAL_DELAY']) if pd.notnull(row['DEPARTURE_DELAY']) and pd.notnull(row['ARRIVAL_DELAY']) else None,
                    "cancelled": int(row['CANCELLED']) if pd.notnull(row['CANCELLED']) else None,
                    "diverted": int(row['DIVERTED']) if pd.notnull(row['DIVERTED']) else None,
                    "air_time": float(row['AIR_TIME']) if pd.notnull(row['AIR_TIME']) else None
                },
                
                "flight_log": None 
            }
            
            # --- MODIFICA PER SCRIVERE JSON VALIDO ---
            if not is_first_record:
                f_out.write(',\n') # 2. Aggiungi virgola e a capo PRIMA del nuovo oggetto (se non è il primo)
            else:
                is_first_record = False # Dopo il primo giro, il flag diventa False

            f_out.write(json.dumps(doc)) # Scriviamo l'oggetto
            
            count += 1

        # Aggiornamento progresso fuori dal loop riga, ma dentro il chunk per non intasare la console
        print(f"   ...processati {count} voli", end='\r')
            
    f_out.write(']') # 3. Chiudiamo l'array JSON alla fine del file

print(f"\n   -> Completato! Creato {OUTPUT_FLIGHTS} con {count} documenti.")

2. Elaborazione Voli e Denormalizzazione (Questo richiederà tempo)...
   ...processati 5262834 voli
   -> Completato! Creato ../dataset/json/flights.json con 5262834 documenti.
